# CNN/RNN Hybrid Model Runner

---
## 1 · Settings


In [ ]:
import torch

# ── Experiment Configurations ─────────────────────────────────────────────────
# List of model/hyperparameter sets to run consecutively.
EXPERIMENTS = [
    {
        "name":              "GRU | log_spectrogram",
        "model":             "cnn_rnn_ctc",
        "rnn_num_layers":    2,
        "rnn_hidden_size":   384,
        "rnn_bidirectional": True,
        "transforms":        "log_spectrogram",
    },
    {
        "name":              "GRU | log_spectrogram_plus",
        "model":             "cnn_rnn_ctc",
        "rnn_num_layers":    2,
        "rnn_hidden_size":   384,
        "rnn_bidirectional": True,
        "transforms":        "log_spectrogram_plus",
    },
    {
        "name":              "LSTM | log_spectrogram",
        "model":             "cnn_lstm_ctc",
        "rnn_num_layers":    2,
        "rnn_hidden_size":   384,
        "rnn_bidirectional": True,
        "transforms":        "log_spectrogram",
    },
    {
        "name":              "LSTM | log_spectrogram_plus",
        "model":             "cnn_lstm_ctc",
        "rnn_num_layers":    2,
        "rnn_hidden_size":   384,
        "rnn_bidirectional": True,
        "transforms":        "log_spectrogram_plus",
    },
]

# ── Global Settings ──────────────────────────────────────────────────────────
BATCH_SIZE = 32           # reduce to 16 if OOM
USER       = "single_user"

# ── Hardware (auto-detected) ───────────────────────────────────────────────────
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES     = torch.cuda.device_count() if torch.cuda.is_available() else 1

print(f"Accelerator  : {ACCELERATOR}")
print(f"Devices      : {DEVICES}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"Experiments  : {len(EXPERIMENTS)}")
for e in EXPERIMENTS:
    print(f"  - {e['name']}")

---
## 2 · Full Training

Runs each experiment in `EXPERIMENTS` consecutively. After each run, `finalize_run.py`
automatically saves `training_curves.png` and `experiment_summary.json` into the run directory.

In [ ]:
for exp in EXPERIMENTS:
    name = exp["name"]
    model = exp["model"]
    
    print(f"\n{'='*80}")
    print(f"RUNNING EXPERIMENT: {name}")
    print(f"{'='*80}\n")
    
    # Run Training
    !python -m emg2qwerty.train \
      model={model} user={USER} \
      transforms={exp['transforms']} \
      trainer.accelerator={ACCELERATOR} trainer.devices={DEVICES} \
      batch_size={BATCH_SIZE} \
      module.rnn_num_layers={exp['rnn_num_layers']} \
      module.rnn_hidden_size={exp['rnn_hidden_size']} \
      module.rnn_bidirectional={exp['rnn_bidirectional']}
    
    # Automatically find logs and save summary for this run
    print(f"\nFinalizing results for {name}...")
    %run -i 'scripts/finalize_run.py' --name "{name}"

---
## 3 · Evaluation

The training run already tests with **greedy decoding** on the best checkpoint automatically.
Use these cells only to run **beam search**, or to re-evaluate an old checkpoint.

**Note:** Either paste your checkpoint path into `CHECKPOINT` in Settings,  
or run the auto-finder cell below to pick up the latest best checkpoint from `logs/`.

In [ ]:
import glob, os

# Auto-find the best checkpoint from the most recent training run.
ckpt_candidates = sorted(
    glob.glob("logs/*/*/checkpoints/*.ckpt"),
    key=os.path.getmtime
)
best_ckpts = [p for p in ckpt_candidates if "last" not in os.path.basename(p)]
CHECKPOINT = best_ckpts[-1] if best_ckpts else ckpt_candidates[-1] if ckpt_candidates else ""

# Normalize to forward slashes — Hydra misparses backslashes and '=' in filenames
CHECKPOINT = CHECKPOINT.replace("\\", "/")

# Default MODEL to the last experiment; override manually if evaluating a different run.
MODEL = EXPERIMENTS[-1]["model"]

print(f"Using checkpoint : {CHECKPOINT}")
print(f"Using model      : {MODEL}")
assert os.path.isfile(CHECKPOINT), f"Checkpoint not found: {CHECKPOINT}"

In [ ]:
# Greedy Decoding — only needed to re-evaluate an old checkpoint
# (training already runs this automatically on the best checkpoint)
!python -m emg2qwerty.train train=False \
  model={MODEL} user={USER} \
  trainer.accelerator={ACCELERATOR} trainer.devices={DEVICES} \
  "checkpoint='{CHECKPOINT}'"

In [ ]:
# Beam Search + Language Model — best accuracy
!python -m emg2qwerty.train train=False \
  model={MODEL} user={USER} \
  trainer.accelerator={ACCELERATOR} trainer.devices={DEVICES} \
  decoder=ctc_beam \
  "checkpoint='{CHECKPOINT}'"

---
## 4 · Training Curves

Reads TensorBoard event files from the most recent training run and plots loss and CER over epochs.

In [ ]:
import glob, os
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# ── Find the most recent training run ─────────────────────────────────────────
log_dirs = glob.glob("logs/*/*/lightning_logs/version_*")
assert log_dirs, "No TensorBoard logs found. Run training first."
log_dir = max(log_dirs, key=os.path.getmtime)   # most recently written

ea = EventAccumulator(log_dir)
ea.Reload()

val_cer_events = ea.Scalars("val/CER")
steps_per_epoch = val_cer_events[-1].step / max(1, len(val_cer_events) - 1)

def load_metric(tag):
    if tag not in ea.Tags()["scalars"]:
        return [], []
    events = ea.Scalars(tag)
    epochs = [e.step / steps_per_epoch for e in events]
    return epochs, [e.value for e in events]

def clip(vals, lo, hi):
    if lo is not None:
        vals = [max(v, lo) for v in vals]
    if hi is not None:
        vals = [min(v, hi) for v in vals]
    return vals

# ── Data ──────────────────────────────────────────────────────────────────────
raw = {
    "Loss":     {k: load_metric(t) for k, t in [("train", "train/loss"), ("val", "val/loss")]},
    "CER":      {k: load_metric(t) for k, t in [("train", "train/CER"),  ("val", "val/CER")]},
    "Accuracy": {k: (e, [100 - v for v in vals])
                 for k, (e, vals) in
                 {k: load_metric(t) for k, t in [("train", "train/CER"), ("val", "val/CER")]}.items()},
}

# (linear_lo, linear_hi, log_lo)  — log_lo prevents log(0/-inf)
bounds = {
    "Loss":     (0,    None, 1e-2),
    "CER":      (0,    100,  1e-1),
    "Accuracy": (0,    100,  1e-1),
}
ylabels = {"Loss": "CTC Loss", "CER": "CER (%)", "Accuracy": "Accuracy (%)"}
colors  = {"train": "steelblue", "val": "darkorange"}

# ── 2×3 grid: linear (top) / log-scale (bottom) ───────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("CNN/RNN Hybrid — Training Curves", fontsize=13)

for col, name in enumerate(["Loss", "CER", "Accuracy"]):
    lin_lo, lin_hi, log_lo = bounds[name]
    for row, log_scale in enumerate([False, True]):
        ax = axes[row][col]
        for split, (epochs, vals) in raw[name].items():
            if not vals:
                continue
            v = clip(vals, log_lo if log_scale else lin_lo, lin_hi)
            ax.plot(epochs, v, label=split, color=colors[split], linewidth=1.2)
        ax.set_title(f"{name}{' (log scale)' if log_scale else ''}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabels[name])
        if log_scale:
            ax.set_yscale("log")
        else:
            ax.set_ylim(lin_lo, lin_hi)
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to training_curves.png")

---
## 5 · Experiment Comparison

Reads all `experiment_summary.json` files across every run and shows a sorted comparison table.
The best test/CER is highlighted in green. Also saves `experiments_comparison.csv` to the project root.

In [ ]:
import glob, json
import pandas as pd
from IPython.display import display

rows = []
for path in sorted(glob.glob("logs/*/*/experiment_summary.json")):
    with open(path) as f:
        s = json.load(f)
    hp = s.get("hyperparameters", {})
    rows.append({
        "run_name":      s.get("run_name", "—"),
        "model":         s.get("model_type", "—"),
        "layers":        hp.get("rnn_num_layers"),
        "hidden":        hp.get("rnn_hidden_size"),
        "bidir":         hp.get("rnn_bidirectional"),
        "batch":         hp.get("batch_size"),
        "lr":            hp.get("lr"),
        "best_val_CER":  s.get("best_val_CER"),
        "test_CER":      s.get("test_CER"),
        "test_loss":     s.get("test_loss"),
        "run_id":        s.get("run_id"),
    })

assert rows, "No experiment_summary.json files found. Run training first."

df = (pd.DataFrame(rows)
        .sort_values("test_CER")
        .reset_index(drop=True))

display(df.style
          .highlight_min(subset=["test_CER", "best_val_CER"], color="lightgreen")
          .format({c: "{:.2f}" for c in ["best_val_CER","test_CER","test_loss"]},
                  na_rep="—"))

df.to_csv("experiments_comparison.csv", index=False)
print(f"\n{len(df)} run(s) — saved to experiments_comparison.csv")